# Back-translation of the 500

In [1]:
# Cell 1. Install (about 2 minutes). transformers 4.46.3 is the version IndicTrans2 ran on in Panel 1.
!pip -q install "transformers==4.46.3" accelerate sentencepiece pandas IndicTransToolkit
print("installed. If Colab asks to restart the session, do it, then continue from cell 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

In [3]:
# Cell 2. Upload two files: 500_translated_gptoss.json and bangla_qe_pipeline.py
from google.colab import files
import os
up = files.upload()
for k in up:
    print(k, os.path.getsize(k), "bytes")
assert os.path.exists("bangla_qe_pipeline.py"), "bangla_qe_pipeline.py is missing: the notebook uses its nfc(), clean_bangla() and segment_labelled()"


Saving 500_translated_gptoss_pass2.json.json to 500_translated_gptoss_pass2.json (1).json
Saving bangla_qe_pipeline.py to bangla_qe_pipeline.py
500_translated_gptoss_pass2.json (1).json 2254530 bytes
bangla_qe_pipeline.py 21549 bytes


In [4]:
# Cell 3. Settings. Change nothing unless the run itself changes.
INPUT_JSON = "500_translated_gptoss_pass2.json"
RUN_NAME = "gptoss120b_pass2_500"
OUT_CSV = f"backtranslations_{RUN_NAME}.csv"

MODEL_NAME = "ai4bharat/indictrans2-indic-en-1B"
SRC_LANG, TGT_LANG = "ben_Beng", "eng_Latn"
NUM_BEAMS = 5
MAX_NEW_TOKENS = 512           # 21 Sep 2026: was 256; long lab lists were cut on the output side
LENGTH_PENALTY = 1.0
REPETITION_PENALTY = 1.0
SEED = 42
BATCH_SIZE = 16
MAX_WORDS = 40                  # 21 Sep 2026: a Bangla sentence longer than this is split at ; and , before translation
RETRY_BELOW = 0.70              # 21 Sep 2026: an item whose back-translation is shorter than this share of the source is redone with 20-word chunks

USE_DRIVE = True          # True: the CSV lives in Google Drive and survives a disconnect. False: /content only.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/bangla_gap"
else:
    OUT_DIR = "/content"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, OUT_CSV)

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("output file:", OUT_PATH)

Mounted at /content/drive
output file: /content/drive/MyDrive/bangla_gap/backtranslations_gptoss120b_pass2_500.csv


In [5]:
# Cell 4. HuggingFace login, then load IndicTrans2 (about 1 minute on T4).
# ai4bharat/indictrans2-indic-en-1B is a gated repo. Without a login the load fails with a
# "couldn't connect to huggingface.co" or a 401 error. Paste a read token from huggingface.co/settings/tokens.
from huggingface_hub import login
login(new_session=False)      # asks for the token once per runtime; a saved token is reused
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()
ip = IndicProcessor(inference=True)
print("loaded on", DEVICE)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

loaded on cuda


In [9]:
# Cell 5. Helpers. The Bangla goes through the same three steps the scoring pipeline uses:
# nfc() (the Panel 1 encoding fix), clean_bangla() (drops the ITEM_ID and QUESTION wrapper lines), segment_labelled() (stem and options).
# 21 Sep 2026: sentences longer than MAX_WORDS are split at ; and , so IndicTrans2 never truncates an input, and an item whose
# back-translation is under RETRY_BELOW of the source length is redone with 20-word chunks (the longer result is kept).
import re, json
import bangla_qe_pipeline as p

def chunk_long(s, max_words):
    words = s.split()
    if len(words) <= max_words:
        return [s]
    parts = re.split(r"(?<=[;,\u0964])\s+", s)
    out, cur = [], ""
    for part in parts:
        if cur and len((cur + " " + part).split()) > max_words:
            out.append(cur.strip()); cur = part
        else:
            cur = (cur + " " + part).strip()
    if cur:
        out.append(cur.strip())
    final = []
    for c in out:                                   # anything still too long is cut by word count
        w = c.split()
        for i in range(0, len(w), max_words):
            final.append(" ".join(w[i:i + max_words]))
    return final

def to_sentences(text, max_words):
    out = []
    for line in text.split("\n"):
        for s in re.split(r"(?<=[\u0964\?!])\s*", line):
            s = s.strip()
            if s:
                out += chunk_long(s, max_words)
    return out

def translate(sents):
    out = []
    for i in range(0, len(sents), BATCH_SIZE):
        chunk = sents[i:i + BATCH_SIZE]
        batch = ip.preprocess_batch(chunk, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
        enc = tok(batch, truncation=True, padding="longest", return_tensors="pt", return_attention_mask=True).to(DEVICE)
        with torch.inference_mode():
            gen = model.generate(**enc, num_beams=NUM_BEAMS, num_return_sequences=1, max_new_tokens=MAX_NEW_TOKENS,
                                 length_penalty=LENGTH_PENALTY, repetition_penalty=REPETITION_PENALTY, early_stopping=True)
        dec = tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        out += ip.postprocess_batch(dec, lang=TGT_LANG)
    return out

def backtranslate_text(bangla_raw, max_words):
    bn, _ = p.nfc(bangla_raw)
    bn = p.clean_bangla(bn)
    stem, opts = p.segment_labelled(bn)
    sents = to_sentences(stem, max_words)
    en_stem = translate(sents) if sents else []
    en_opts = translate([t for _, t in opts]) if opts else []
    tail = "\n".join(f"{lab}. {en}" for (lab, _), en in zip(opts, en_opts))       # the model's own labels, with a full stop like the source
    return (" ".join(en_stem) + "\n" + tail).strip(), len(sents)

def backtranslate_record(r):
    """Returns (back_en, n_chunks, retry, len_ratio). Redoes the item with 20-word chunks when the first pass is short."""
    src_len = max(len(r["english"]), 1)
    en, n = backtranslate_text(r["bangla"], MAX_WORDS)
    retry = 0
    if len(en) / src_len < RETRY_BELOW:
        en2, n2 = backtranslate_text(r["bangla"], 20)
        if len(en2) > len(en):
            en, n, retry = en2, n2, 1
    return en, n, retry, round(len(en) / src_len, 3)

rec0 = json.load(open(INPUT_JSON, encoding="utf-8"))[0]
en0, n0, retry0, ratio0 = backtranslate_record(rec0)
print(f"{rec0['item_id']}: {n0} chunks, retry {retry0}, length ratio {ratio0}")
print(en0[:800])

US-00001: 9 chunks, retry 0, length ratio 0.939
The 4670-gram (10-pound 5-ounce) male neonate Terme delivered after prolonged labor to a 26-year-old female. The Apgar score is 9 and 9 at both 1 and 5 minutes. Delivery room tests show swelling, sensitivity, and cryptis above the left clavicle. The left upper limb has slowed down. Movement of the hands and wrists is normal. Grass reflex is normal in both hands. An asymmetric Moro reflex is present. The rest of the test shows no abnormalities, and an anteroposterior X-ray confirms the diagnosis. Which of the following is the most appropriate next step in management?
A. Nerve conductivity test
B. Surgical fixation
C. Physical therapy
D. Pin Sleeve on Shirt
E. arm splinting
F. MRI of the clavicle


In [10]:
# Cell 6. Back-translate all 500. Writes after every item and resumes if re-run.
import csv, json, time
import pandas as pd

recs = json.load(open(INPUT_JSON, encoding="utf-8"))
done = set()
if os.path.exists(OUT_PATH) and os.path.getsize(OUT_PATH) > 0:
    done = set(pd.read_csv(OUT_PATH, encoding="utf-8-sig")["item_id"].astype(str))
    print("resuming; already done:", len(done))
t_start = time.time()
n_new = 0
with open(OUT_PATH, "a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["item_id", "translator", "back_en", "seconds", "n_chunks", "retry", "len_ratio"])
    if not done:
        w.writeheader()
    for i, r in enumerate(recs, 1):
        if r["item_id"] in done or r.get("error") or not str(r.get("bangla") or "").strip():
            continue
        t0 = time.time()
        en, n_chunks, retry, ratio = backtranslate_record(r)
        w.writerow({"item_id": r["item_id"], "translator": r.get("model", "gpt-oss:120b"), "back_en": en,
                    "seconds": round(time.time() - t0, 2), "n_chunks": n_chunks, "retry": retry, "len_ratio": ratio})
        f.flush()
        n_new += 1
        if n_new % 25 == 0:
            print(f"{i} of {len(recs)}: {(time.time() - t_start) / 60:.1f} min so far")
print(f"finished: {n_new} new records in {(time.time() - t_start) / 60:.1f} min")

25 of 500: 0.6 min so far
50 of 500: 1.2 min so far
75 of 500: 1.8 min so far
100 of 500: 2.4 min so far
125 of 500: 3.0 min so far
150 of 500: 3.6 min so far
175 of 500: 4.1 min so far
200 of 500: 4.7 min so far
225 of 500: 5.4 min so far
250 of 500: 6.1 min so far
275 of 500: 6.7 min so far
300 of 500: 7.3 min so far
325 of 500: 8.0 min so far
350 of 500: 8.6 min so far
375 of 500: 9.3 min so far
400 of 500: 9.9 min so far
425 of 500: 10.5 min so far
450 of 500: 11.0 min so far
475 of 500: 11.6 min so far
500 of 500: 12.2 min so far
finished: 500 new records in 12.2 min


In [11]:
# Cell 7. Throughput, the length check, and download. Report the numbers printed here.
df = pd.read_csv(OUT_PATH, encoding="utf-8-sig")
df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")            # BOM so Excel opens it correctly
sec = df["seconds"]
print(f"{len(df)} records. Total {sec.sum() / 60:.1f} min. {sec.mean():.1f} s per item. "
      f"Projection for 12,000 items: {sec.mean() * 12000 / 3600:.1f} h on a T4.")
print("empty back-translations:", int((df["back_en"].fillna("").str.strip() == "").sum()))
print("length ratio (back-translation / English source), characters:")
print(df["len_ratio"].describe().round(2))
print("items under 0.70 after the retry:", int((df["len_ratio"] < 0.70).sum()), "| items that needed the retry:", int(df["retry"].sum()))
print(df.nsmallest(8, "len_ratio")[["item_id", "len_ratio", "retry", "n_chunks"]].to_string(index=False))
print("---", df["item_id"].iloc[0]); print(df["back_en"].iloc[0][:600])
files.download(OUT_PATH)

500 records. Total 12.2 min. 1.5 s per item. Projection for 12,000 items: 4.9 h on a T4.
empty back-translations: 0
length ratio (back-translation / English source), characters:
count    500.00
mean       1.03
std        0.04
min        0.87
25%        1.00
50%        1.02
75%        1.05
max        1.25
Name: len_ratio, dtype: float64
items under 0.70 after the retry: 0 | items that needed the retry: 0
 item_id  len_ratio  retry  n_chunks
US-00328      0.869      0         8
US-00254      0.916      0        11
US-00001      0.939      0         9
US-00401      0.939      0         3
US-00079      0.940      0        10
US-00388      0.940      0         7
US-00170      0.944      0        10
US-00025      0.946      0        11
--- US-00001
The 4670-gram (10-pound 5-ounce) male neonate Terme delivered after prolonged labor to a 26-year-old female. The Apgar score is 9 and 9 at both 1 and 5 minutes. Delivery room tests show swelling, sensitivity, and cryptis above the left clavicle. T

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>